In [1]:
# Décommenter la ligne ci-dessous uniquement si tu exécutes ce notebook sur Google Colab :
# !pip install -q transformers accelerate bitsandbytes gradio

import gc
import time
import torch
import gradio as gr
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

cache_modele = {"cle": None, "processor": None, "model": None}

def generer_legende(image, model_id, mode):
    if image is None:
        return "Veuillez charger une image.", "-", "-"

    image = image.convert("RGB")
    image.thumbnail((512, 512))
    cle_actuelle = f"{model_id}_{mode}"

    if cache_modele["cle"] != cle_actuelle:
        cache_modele["processor"] = None
        cache_modele["model"] = None
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        if mode == "4bit":
            quant_config = BitsAndBytesConfig(load_in_4bit=True)
        elif mode == "8bit":
            quant_config = BitsAndBytesConfig(load_in_8bit=True)
        else:
            quant_config = None

        cache_modele["processor"] = AutoProcessor.from_pretrained(model_id)
        cache_modele["model"] = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype=torch.float16, quantization_config=quant_config, device_map="auto"
        )
        cache_modele["cle"] = cle_actuelle

    processor = cache_modele["processor"]
    model = cache_modele["model"]

    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Describe this image clearly in one or two sentences."}
    ]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], padding=True, return_tensors="pt").to("cuda")

    debut = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=60)
    duree = time.time() - debut

    reponse_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    legende = processor.batch_decode(reponse_ids, skip_special_tokens=True)[0].strip()
    vram_go = torch.cuda.max_memory_allocated() / (1024 ** 3)

    return legende, f"{duree:.2f} secondes", f"{vram_go:.2f} Go"

interface = gr.Interface(
    fn=generer_legende,
    inputs=[
        gr.Image(type="pil", label="1. Charger une image"),
        gr.Dropdown(choices=["HuggingFaceTB/SmolVLM-Instruct", "Qwen/Qwen2-VL-2B-Instruct"], value="HuggingFaceTB/SmolVLM-Instruct", label="2. Modèle VLM"),
        gr.Radio(choices=["fp16", "8bit", "4bit"], value="4bit", label="3. Quantification")
    ],
    outputs=[
        gr.Textbox(label="Légende générée", lines=3),
        gr.Textbox(label="Temps d'inférence"),
        gr.Textbox(label="VRAM maximale")
    ],
    title="Projet 7 — Comparateur de VLMs quantifiés"
)

interface.launch(share=True)

c:\Users\USER\Desktop\PRJ7-VLM\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1125 21:55:30.022000 27996 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
